# ML Framework - Jupyter Demo

This notebook demonstrates how to use the ML Framework managers in Jupyter notebooks.

## Features:
- Data loading and feature generation
- Feature importance analysis with inline visualization
- Model training and evaluation
- Backtest visualization
- Interactive Plotly charts

In [ ]:
# Setup
import sys
from pathlib import Path

# Add parent directory to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Import ML Framework components
from src.data_provider import DataProvider
from src.features_generator import FeaturesGenerator
from src.models_lib import RandomForestModel, XGBoostModel, LogisticRegressionModel
from src.backtesting import BacktestNoLib
from src.managers.visualization_manager import VisualizationManager
from src.managers.result_manager import ResultManager
from src.managers.feature_selector import FeatureSelector
from src.managers.run_manager import RunManager
from src.managers.scaler_manager import ScalerManager

## 1. Load Data

In [ ]:
# Configuration
TICKER = 'BTC-USD'
START_DATE = '2022-01-01'
END_DATE = '2024-11-24'
FUTURE_BARS = 15
THRESHOLD = 0.05

# Load data
data_provider = DataProvider(data_dir='../data')
df = data_provider.load_yahoo(ticker=TICKER, start_date=START_DATE, end_date=END_DATE, use_cache=True)

print(f"Loaded {len(df)} rows")
df.head()

## 2. Generate Features

In [ ]:
# Generate features
features_gen = FeaturesGenerator()
df_features = features_gen.generate_features(df, feature_set='advanced')
df_features = features_gen.create_target(
    df_features,
    target_type='classification',
    future_bars=FUTURE_BARS,
    threshold=THRESHOLD,
    num_classes=3
)
df_features = df_features.dropna()

feature_cols = features_gen.get_feature_names()
print(f"Generated {len(feature_cols)} features")
print(f"Dataset: {len(df_features)} rows")

# Show target distribution
df_features['target'].value_counts().sort_index()

## 3. Split Data

In [ ]:
# Temporal split
train_df, val_df, test_df = data_provider.split_data(df_features, train_ratio=0.7, val_ratio=0.15)

X_train = train_df[feature_cols].values
y_train = train_df['target'].values
X_val = val_df[feature_cols].values
y_val = val_df['target'].values
X_test = test_df[feature_cols].values
y_test = test_df['target'].values

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

## 4. Feature Importance Analysis (Jupyter Display)

In [ ]:
# Initialize managers
viz_manager = VisualizationManager()  # Auto-detects Jupyter
feature_selector = FeatureSelector(method='tree')

# Fit feature selector
feature_selector.fit(train_df[feature_cols], train_df['target'])

# Get feature importance
feature_importance = feature_selector.get_feature_importance()
selected_features = feature_selector.get_selected_features()
dropped_features = feature_selector.get_dropped_features()

print(f"Selected: {len(selected_features)}, Dropped: {len(dropped_features)}")

In [ ]:
# Display feature importance inline (Jupyter-friendly)
# Method 1: Show all figures at once
viz_manager.show_feature_importance(
    feature_importance=feature_importance,
    selected_features=selected_features,
    dropped_features=dropped_features,
    method='tree'
)

In [ ]:
# Method 2: Get figures and display individually
figures = viz_manager.get_feature_importance_figures(
    feature_importance=feature_importance,
    selected_features=selected_features,
    dropped_features=dropped_features,
    method='tree'
)

# Display just the first figure (top features bar chart)
figures[0].show()

## 5. Train Models

In [ ]:
# Scale features
scaler_manager = ScalerManager(scaler_type='standard')
X_train_scaled = scaler_manager.fit_transform(X_train)
X_val_scaled = scaler_manager.transform(X_val)
X_test_scaled = scaler_manager.transform(X_test)

# Create models
models = {
    'LogisticRegression': LogisticRegressionModel(name='LogisticRegression', max_iter=1000),
    'RandomForest': RandomForestModel(name='RandomForest', n_estimators=100, max_depth=10),
}

# Try to add XGBoost
try:
    models['XGBoost'] = XGBoostModel(name='XGBoost', n_estimators=100, max_depth=6)
except:
    print("XGBoost not available")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    
    # Calculate all metrics with zero_division=0 to handle missing classes
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    results[name] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'status': 'success'
    }
    
    print(f"  Accuracy:  {accuracy:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1:        {f1:.4f}")
    
    # Show detailed classification report
    print(f"\n  Classification Report for {name}:")
    print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
# Per-class recall analysis (important for trading signals)
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd

print("="*70)
print("PER-CLASS RECALL ANALYSIS")
print("="*70)
print("\nRecall = TP / (TP + FN) = How many actual positives did we catch?")
print("High recall for class +1 (UP) means we catch most upward movements.\n")

# Get unique classes from y_test
unique_classes = sorted(np.unique(y_test))
class_names = {-1: 'DOWN', 0: 'FLAT', 1: 'UP'}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_test, y_pred, labels=unique_classes, zero_division=0
    )
    
    print(f"\n{name}:")
    print("-" * 50)
    
    metrics_df = pd.DataFrame({
        'Class': [class_names.get(c, f'Class {c}') for c in unique_classes],
        'Label': unique_classes,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Support': support.astype(int)
    })
    
    print(metrics_df.to_string(index=False))
    
    # Summary for trading - find UP class (label=1)
    if 1 in unique_classes:
        up_idx = list(unique_classes).index(1)
        print(f"\n  → UP class recall: {recall[up_idx]:.4f} (catching {recall[up_idx]*100:.1f}% of upward moves)")
        print(f"  → UP class precision: {precision[up_idx]:.4f} (when predicting UP, correct {precision[up_idx]*100:.1f}% of time)")
    
    # Also show DOWN class for short opportunities
    if -1 in unique_classes:
        down_idx = list(unique_classes).index(-1)
        print(f"  → DOWN class recall: {recall[down_idx]:.4f} (catching {recall[down_idx]*100:.1f}% of downward moves)")

In [ ]:
# Display results comparison (Jupyter-friendly)
result_manager = ResultManager()
result_manager.add_test_results(results)

# This will display a styled DataFrame in Jupyter
result_manager.display_test_comparison()

## 6. Run Backtests

In [ ]:
# Backtest configuration
INITIAL_CAPITAL = 10000.0
COMMISSION = 0.001
POSITION_SIZE = 0.02
BARS_TO_HOLD = FUTURE_BARS

# Prepare backtest data
backtest_df = df_features.copy()
for col in ['open', 'high', 'low', 'close', 'volume']:
    if col not in backtest_df.columns and col in df.columns:
        backtest_df[col] = df[col]

# Run backtests
backtest_results = {}

for name, model in models.items():
    print(f"\nBacktesting {name}...")
    
    backtest = BacktestNoLib(
        initial_capital=INITIAL_CAPITAL,
        commission=COMMISSION,
        position_size=POSITION_SIZE,
        bars_to_hold=BARS_TO_HOLD
    )
    
    results = backtest.run(
        df=backtest_df,
        model=model,
        scaler=scaler_manager.scaler,
        feature_cols=feature_cols,
        price_col='close'
    )
    
    metrics = backtest.get_metrics()
    print(f"  Return: {metrics['total_return']*100:.2f}%, Sharpe: {metrics['sharpe_ratio']:.2f}")
    
    # Store for visualization
    equity_curve = results.get('equity_curve', [])
    if isinstance(equity_curve, pd.Series):
        equity_curve = equity_curve.tolist()
    
    backtest_results[name] = {
        'equity_curve': equity_curve,
        'trades': backtest.get_trades(),
        'metrics': metrics,
        'initial_capital': INITIAL_CAPITAL
    }

## 7. Visualize Backtest Results (Jupyter Display)

In [ ]:
# Display backtest results inline
viz_manager.show_backtest_results(backtest_results, df=backtest_df)

In [ ]:
# Or get individual figures
figures = viz_manager.get_backtest_figures(backtest_results, df=backtest_df)

# Display equity curves comparison
figures[1].show()  # Equity curves comparison is usually the second figure

## 8. Save Results (Optional)

In [ ]:
# Initialize RunManager to save artifacts
run_manager = RunManager(base_dir='../models')
run_manager.initialize()

# Save feature importance report (HTML file)
viz_manager.create_feature_importance_report(
    feature_importance=feature_importance,
    save_dir=run_manager.get_run_dir(),
    selected_features=selected_features,
    dropped_features=dropped_features,
    method='tree',
    show=False  # Don't open browser
)

print(f"\nArtifacts saved to: {run_manager.get_run_dir()}")

## Summary

### Jupyter-Friendly Methods:

**VisualizationManager:**
- `show_feature_importance()` - Display feature importance charts inline
- `show_backtest_results()` - Display backtest charts inline
- `show_model_comparison()` - Display model comparison charts inline
- `get_*_figures()` - Get list of figures for individual display
- `display_figure()` / `display_figures()` - Display Plotly figures

**ResultManager:**
- `display_train_comparison()` - Styled DataFrame with training results
- `display_test_comparison()` - Styled DataFrame with test results
- `display_backtest_comparison()` - Styled DataFrame with backtest results
- `display_summary()` - HTML formatted summary

All methods auto-detect Jupyter environment and display appropriately.